# G1 Academy Bonus - Task 3: developing say() and set_headlight() from scratch

## Introduction
Building on Task 2's DDS/pub-sub helpers, this task develops full native versions of `say` and `set_headlight`, matching the behavior of `sdk_wrapper.G1.say`/`set_headlight` (not just the shortcut used in the intro academy notebook). We still reuse `util.py`'s Piper subprocess/WAV-conversion boilerplate - that part is intentionally not something to retype - but we build the `AudioClient` plumbing, color parsing, and the cancellable background headlight thread ourselves.

## Task 1 - `AudioClient` + `say()`
`AudioClient` is the native SDK request client for volume, `LedControl`, and `PlayStream`. `util.play_piper_text` does only the tedious part (Piper synthesis, resampling to mono/16-bit/16kHz, `PlayStream` framing); you own constructing and initializing the client.

In [2]:
import sys
import time
sys.path.append("..")
from unitree_sdk2py.core.channel import ChannelFactoryInitialize
from unitree_sdk2py.g1.audio.g1_audio_client import AudioClient
from util import play_piper_text

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

audio_client = AudioClient()
audio_client.SetTimeout(5.0)
audio_client.Init()

def say(text, language="en", volume=100):
    return play_piper_text(audio_client, text, language=language, volume=volume)

print(say("Headlight and speech helpers online."))

0


## Task 2 - Named/hex/rgb color parsing + intensity scaling
`set_headlight(color=...)` accepts a name, a `#RRGGBB` hex string, or an `"R,G,B"` string. Parse each form into a clamped `(r, g, b)` tuple, then scale it by an intensity percentage before it is ever sent to `LedControl`.

In [3]:
import re

_NAMED_COLORS = {
    "white": (255, 255, 255), "red": (255, 0, 0), "green": (0, 255, 0), "blue": (0, 0, 255),
    "yellow": (255, 255, 0), "cyan": (0, 255, 255), "magenta": (255, 0, 255),
    "orange": (255, 165, 0), "purple": (128, 0, 128), "pink": (255, 105, 180),
}

def parse_color(value):
    if isinstance(value, tuple) and len(value) == 3:
        return tuple(int(max(0, min(255, v))) for v in value)
    value = str(value).strip().lower()
    if value in _NAMED_COLORS:
        return _NAMED_COLORS[value]
    if re.fullmatch(r"#?[0-9a-fA-F]{6}", value):
        value = value.lstrip("#")
        return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))
    if re.fullmatch(r"\d{1,3},\d{1,3},\d{1,3}", value):
        return tuple(max(0, min(255, int(x))) for x in value.split(","))
    raise ValueError("color must be a name, #RRGGBB, or R,G,B")

def scale_color(rgb, intensity):
    scale = max(0, min(100, int(intensity))) / 100.0
    return tuple(int(x * scale) for x in rgb)

parse_color("cyan"), parse_color("#00ffff"), parse_color("0,255,255")

((0, 255, 255), (0, 255, 255), (0, 255, 255))

## Task 3 - Cancellable background headlight thread + `set_headlight()`
`LedControl` only sets the color for a moment, so holding a color for `duration_s` needs a thread that refreshes it periodically. The thread must be safely cancellable: a new call to `set_headlight` (a new color, or a mode/controller transition) must stop and join any earlier thread before starting its own, and the thread must always attempt to turn the light off in its `finally` block so a crash does not leave a color stuck on.

In [4]:
import threading
import time

def led_control_was_accepted(code):
    # The G1 audio LED RPC can time out (3104) even when the LED command was applied.
    return int(code) in (0, 3104)

class HeadlightThread(threading.Thread):
    def __init__(self, audio_client, rgb, duration_s, interval_s, stop_event):
        super().__init__(daemon=False)
        self.audio_client, self.rgb = audio_client, rgb
        self.duration_s = max(0.0, float(duration_s))
        self.interval_s = max(0.0, float(interval_s))
        self.stop_event = stop_event
    def run(self):
        end_time = time.monotonic() + self.duration_s
        try:
            while not self.stop_event.is_set() and time.monotonic() < end_time:
                code = int(self.audio_client.LedControl(*self.rgb))
                if not led_control_was_accepted(code):
                    self.stop_event.set()
                    break
                self.stop_event.wait(self.interval_s)
        finally:
            try:
                self.audio_client.LedControl(0, 0, 0)
            except Exception:
                pass

_headlight_stop = None
_headlight_thread = None
def set_headlight(color="green", intensity=100, duration_s=3):
    global _headlight_stop, _headlight_thread
    rgb = scale_color(parse_color(color), intensity)
    if _headlight_thread is not None and _headlight_thread.is_alive():
        _headlight_stop.set()
        _headlight_thread.join()
    code = int(audio_client.LedControl(*rgb))
    if not led_control_was_accepted(code) or float(duration_s) <= 0:
        return code
    _headlight_stop = threading.Event()
    _headlight_thread = HeadlightThread(audio_client, rgb, duration_s, 0.2, _headlight_stop)
    _headlight_thread.start()
    return code

set_headlight("yellow", intensity=100, duration_s=30)

0

You have now reconstructed `sdk_wrapper.G1.say` and `sdk_wrapper.G1.set_headlight` natively - compare this cell against `_HeadlightThread`/`G1.set_headlight` in `sdk_wrapper.py`.

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.